# Medical FAQ Chatbot--( CuraBot🤖)


#### CuraBot is an AI-powered intelligent healthcare chatbot designed to provide instant and accurate answers to medical FAQs.
#### Using NLP and similarity matching,it delivers fast, smart, and reliable responses just like a virtual medical assistant.

### Install and Import Libraries

In [14]:
import pandas as pd
import numpy as np
import nltk
import re
import string
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Download necessary NLTK data
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

print("Libraries imported and NLTK data downloaded successfully.")


Libraries imported and NLTK data downloaded successfully.


[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /usr/share/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


### Load and Prepare the Dataset

In [15]:
#  Dataset Path 
file_path = '/kaggle/input/datasets/pythonafroz/medquad-medical-question-answer-for-ai-research/medquad.csv'

try:
    # Load the dataset
    df = pd.read_csv(file_path)
    
    # We only need 'question' and 'answer' columns
    # Adjust column names if they differ in your specific version of the dataset
    df = df[['question', 'answer']].dropna().reset_index(drop=True)
    
    print(f"Dataset loaded. Total FAQs: {len(df)}")
    print("Sample Question:", df['question'][0])
except Exception as e:
    print(f"Error loading dataset: {e}")
    print("Please ensure the dataset path is correct in Kaggle.")


Dataset loaded. Total FAQs: 16407
Sample Question: What is (are) Glaucoma ?


### NLP Preprocessing Function

In [16]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    # Convert to lowercase
    text = str(text).lower()
    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    # Tokenize
    tokens = word_tokenize(text)
    # Remove stopwords and Lemmatize
    cleaned_tokens = [lemmatizer.lemmatize(w) for w in tokens if w not in stop_words]
    
    return " ".join(cleaned_tokens)

# Preprocess all questions in the dataset
print("Preprocessing questions... this may take a moment.")
df['processed_question'] = df['question'].apply(preprocess_text)
print("Preprocessing complete.")


Preprocessing questions... this may take a moment.
Preprocessing complete.


### Vectorization and Similarity Logic

In [17]:
# Initialize TF-IDF Vectorizer
vectorizer = TfidfVectorizer()

# Fit and transform the processed questions
tfidf_matrix = vectorizer.fit_transform(df['processed_question'])

def get_chatbot_response(user_query):
    # 1. Preprocess user input
    processed_query = preprocess_text(user_query)
    
    # 2. Vectorize user input
    query_vector = vectorizer.transform([processed_query])
    
    # 3. Calculate Cosine Similarity against all questions
    similarity_scores = cosine_similarity(query_vector, tfidf_matrix)
    
    # 4. Get the index of the highest score
    best_match_idx = np.argmax(similarity_scores)
    confidence = similarity_scores[0][best_match_idx]
    
    # 5. Return response if confidence is high enough
    if confidence > 0.2: 
        return df['answer'][best_match_idx], confidence
    else:
        return "I'm sorry, I couldn't find a specific answer to that medical query. Please consult a doctor.", confidence

print("Chatbot engine is ready.")


Chatbot engine is ready.


### Interactive Chat UI (Simple Loop)

In [19]:
def start_chat():
    print("-" * 50)
    print("Medical AI Chatbot (Type 'quit' to exit)")
    print("-" * 50)
    
    while True:
        user_input = input("You: ")
        if user_input.lower() in ['quit', 'exit', 'bye']:
            print("Chatbot: Goodbye! Stay healthy.")
            break
        
        response, score = get_chatbot_response(user_input)
        
        print(f"\nChatbot: {response}")
        print(f"(Confidence Score: {score:.2f})\n")

# Start the interaction
start_chat()


--------------------------------------------------
Medical AI Chatbot (Type 'quit' to exit)
--------------------------------------------------


You:  what is fever?



Chatbot: A fever is a body temperature that is higher than normal. It is not an illness. It is part of your body's defense against infection. Most bacteria and viruses that cause infections do well at the body's normal temperature (98.6 F). A slight fever can make it harder for them to survive. Fever also activates your body's immune system.    Infections cause most fevers. There can be many other causes, including       -  Medicines    -  Heat exhaustion    -  Cancers    -  Autoimmune diseases       Treatment depends on the cause of your fever. Your health care provider may recommend using over-the-counter medicines such as acetaminophen or ibuprofen to lower a very high fever. Adults can also take aspirin, but children with fevers should not take aspirin. It is also important to drink enough liquids to prevent dehydration.
(Confidence Score: 1.00)



You:  quit


Chatbot: Goodbye! Stay healthy.


### Advanced UI---- Optional Work

### Install Gradio

In [20]:
!pip install gradio -q


### Launch the Web Application

In [21]:
import gradio as gr
import tempfile

# 1. Removed theme from Blocks constructor (it is now set automatically or passed later)
with gr.Blocks(title="Med-AI Assistant") as demo:
    
    # --- HEADER ---
    gr.HTML("""
    <div style="text-align: center; margin-bottom: 1rem;">
        <h1 style="color: #2c3e50; font-family: Arial, sans-serif;"> CuraBot🤖</h1>
        <p style="color: #7f8c8d; font-size: 1.1em;">NLP-Powered Medical Knowledge Retrieval System</p>
    </div>
    """)
    
    with gr.Row():
        # --- LEFT SIDEBAR (Controls & Info) ---
        with gr.Column(scale=1, min_width=300):
            
            with gr.Accordion("ℹ️ System Architecture", open=True):
                gr.Markdown("""
                **Dataset:** NIH MedQuAD  
                **Algorithm:** TF-IDF Vectorization  
                **Matching:** Cosine Similarity  
                
                *This system maps user natural language queries to an indexed database of medical FAQs.*
                """)
            
            with gr.Accordion("⚠️ Medical Disclaimer", open=False):
                gr.Markdown("""
                *This chatbot is designed for educational and AI research purposes only. It does not constitute medical advice. Always consult a certified healthcare professional for medical diagnoses.*
                """)
                
            gr.Markdown("### 📊 Metrics")
            confidence_display = gr.Number(label="Algorithm Match Confidence", interactive=False, value=0.0)
            
            gr.Markdown("### 🛠️ Tools")
            export_btn = gr.Button("💾 Download Chat History", variant="secondary")
            file_download = gr.File(label="Your Downloaded History", visible=False)

        # --- RIGHT MAIN AREA (Chat UI) ---
        with gr.Column(scale=3):
            
            chatbot = gr.Chatbot(
                label="Consultation Room", 
                height=500, 
                type="messages",       
                allow_tags=True        
                
            )
            
            with gr.Row():
                msg = gr.Textbox(
                    show_label=False,
                    placeholder="Type your medical query here and press Enter...",
                    scale=4,
                    container=False
                )
                submit_btn = gr.Button("📤 Send", variant="primary", scale=1)
                
            with gr.Row():
                clear_btn = gr.Button("🗑️ Clear Conversation", size="sm")
                undo_btn = gr.Button("↩️ Undo Last Query", size="sm")

    # --- BACKEND LOGIC ---
    
    # State variable for managing history in the new format
    state = gr.State([])

    def respond(user_message, chat_history):
        if not user_message.strip():
            return "", chat_history, chat_history, 0.0
            
        # Get response and score from our ML model
        response, score = get_chatbot_response(user_message)
        
        # 3. Updated data structure to match type='messages'
        new_history = chat_history + [
            {"role": "user", "content": user_message},
            {"role": "assistant", "content": response}
        ]
        return "", new_history, new_history, round(score, 4)

    def undo_last(chat_history):
        
        if len(chat_history) >= 2:
            chat_history = chat_history[:-2]
        return chat_history, chat_history

    def export_history(chat_history):
        if not chat_history:
            return gr.update(visible=False)
            
        # Format the chat history for the text file based on the new dictionary structure
        content = "MED-AI CONSULTATION HISTORY\n" + "="*30 + "\n\n"
        for message in chat_history:
            role = "USER" if message["role"] == "user" else "BOT"
            content += f"{role}: {message['content']}\n"
            
            # Add a separator line after the bot replies
            if message["role"] == "assistant":
                content += "-"*30 + "\n"
            
        # Create a temporary file to download
        with tempfile.NamedTemporaryFile(delete=False, mode="w", suffix=".txt") as f:
            f.write(content)
            temp_path = f.name
            
        return gr.update(value=temp_path, visible=True)

    # --- EVENT LISTENERS ---
    
    msg.submit(respond, inputs=[msg, state], outputs=[msg, chatbot, state, confidence_display])
    submit_btn.click(respond, inputs=[msg, state], outputs=[msg, chatbot, state, confidence_display])
    
    # 4. Clear state correctly
    clear_btn.click(lambda: ([], [], 0.0), None, [chatbot, state, confidence_display], queue=False)
    
    undo_btn.click(undo_last, inputs=[state], outputs=[chatbot, state])
    export_btn.click(export_history, inputs=[state], outputs=[file_download])

# Launch the app 
demo.launch(share=True)


* Running on local URL:  http://127.0.0.1:7863
* Running on public URL: https://5097fff3f7c95ce5c0.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
